# Project C · School commute reliability and transportation access
**Audriana Duvall · Data Science for Lawyers**

**Question.** During school commute hours, were delays and cancellations more common at Baltimore City stops in census tracts with higher poverty or lower household vehicle access?

This analysis measures service at stop locations. Household vehicle availability describes local transportation options. Student boarding, home locations, and arrival at school are outside the data. [MTA student transit program](https://www.mta.maryland.gov/student-fares) · [MTA developer resources](https://www.mta.maryland.gov/developer-resources)

**What this notebook produces.** Running every box (Runtime → Run all) does the whole project from raw records to finished page:
1. Cleans and analyzes the MTA stop-event records and joins them to Census tracts and city neighborhoods.
2. Exports the page's data file, `Baltimore_City_MTA_Map_Data.json`.
3. Draws the neighborhood map and the poverty comparison chart right here in Colab.
4. Writes the finished interactive page, `Baltimore_City_MTA_Reliability_Map.html`, with the fresh data embedded, and shows it below the last box.

The repository revision is pinned, so the results reproduce October 30–November 20, 2026.

## 1. Sources and reproducibility
Sources: [Project C stop-event records](https://github.com/TbilKitty/MTA_Accountability_Tracker/tree/bea8c62f6565245e81c7dd80e3c77470e48b454d/data/school-commute-raw), [2024 TIGER/Line Maryland tracts](https://catalog.data.gov/dataset/tiger-line-shapefile-2024-state-maryland-md-census-tract), [Baltimore City neighborhood boundaries](https://baltegis.baltimorecity.gov/server/rest/services/CityView/Neighborhoods/FeatureServer), and 2020–2024 ACS tables [B17001, poverty](https://data.census.gov/table/ACSDT5Y2024.B17001) and [B08201, vehicle availability](https://data.census.gov/table/ACSDT5Y2024.B08201), retrieved through [Census Reporter](https://censusreporter.org/).

### Box 1 · Install and import the tools
`pyshp` reads the Census shapefile and `shapely` does the point-in-polygon and overlap geometry. Colab already has pandas and matplotlib.

In [ ]:
# Standard-library tools for files, downloads and running git.
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

# Install the two geometry libraries only if Colab does not already have them.
if any(importlib.util.find_spec(name) is None for name in ("shapefile", "shapely")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyshp>=2.3,<4",
                    "shapely>=2,<3"], check=True)

import pandas as pd                      # tables
import shapefile                         # reads the Census .shp/.dbf files
from shapely import Point, STRtree, make_valid
from shapely.geometry import shape       # turns GeoJSON-style dicts into shapes
from IPython.display import display, HTML

### Box 2 · Pin the transit records and get the other inputs
The commit SHA fixes the transit sample for grading, so anyone who runs this gets the same numbers. To see fresh numbers instead, set `USE_LATEST = True` at the top of the box: the tracker collects new school days automatically, and every box after this one, including the finished page, will use them. The ACS release and boundary vintage are fixed by their URLs.

Nothing needs uploading: the box downloads the MTA records from the public GitHub repository and the boundary and Census files from their official sources. If a government server is down, the box says which file failed; save that file from your browser, upload it to the Colab Files pane under the name shown, and run the box again.

In [ ]:
# SWITCH: False = the fixed Oct 30 - Nov 20, 2026 sample used for the class submission.
#         True  = every school day collected so far (the tracker adds new days automatically).
USE_LATEST = False

revision = "main" if USE_LATEST else "bea8c62f6565245e81c7dd80e3c77470e48b454d"   # pinned commit
root = Path(tempfile.mkdtemp(prefix="mta-equity-"))       # scratch folder for the inputs

# Official download address for each non-MTA input.
inputs = {
    "neighborhoods.json": ("https://baltegis.baltimorecity.gov/server/rest/services/CityView/"
                           "Neighborhoods/FeatureServer/0/query?where=1%3D1"
                           "&outFields=OBJECTID%2CName&outSR=4326&f=geojson"),
    "tract.zip": "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_24_tract.zip",
    "reporter.json": ("https://api.censusreporter.org/1.0/data/show/acs2024_5yr"
                      "?table_ids=B17001,B08201&geo_ids=140%7C05000US24510"),
}
# File names to look for if you uploaded a copy yourself (Files pane or a ZIP).
aliases = {"neighborhoods.json": ["neighborhoods.json", "*eighborhood*.json", "*eighborhood*.geojson"],
           "tract.zip": ["tract.zip", "tl_2024_24_tract.zip"],
           "reporter.json": ["reporter.json"]}

# 1) Optional: use copies you uploaded, loose or inside Project_C_Inputs.zip.
search_dirs = [Path(".")]
bundle = next(iter(sorted(Path(".").glob("Project_C_Inputs*.zip"))), None)
if bundle:
    zipfile.ZipFile(bundle).extractall(root / "bundle")
    search_dirs.append(root / "bundle")
    print(f"Using files from {bundle.name}")
for name, patterns in aliases.items():
    found = next((p for d in search_dirs for pat in patterns for p in sorted(d.rglob(pat))), None)
    if found:
        (root / name).write_bytes(found.read_bytes())
        print(f"Using uploaded {found.name} for {name}")

# 2) Fetch the tracker records at the pinned commit (one snapshot, not the whole history).
#    MTA_REPO_DIR is only set by the website's automatic rebuild, which already has the files.
if os.environ.get("MTA_REPO_DIR"):
    repo = Path(os.environ["MTA_REPO_DIR"])
else:
    repo = root / "tracker"
    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "-q", "--depth", "1",
                    "https://github.com/TbilKitty/MTA_Accountability_Tracker.git", revision], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)

# 3) Download anything not uploaded. Some government servers refuse Python's default
#    identity, so the request names itself like a browser, and each download gets 3 tries.
def download(url, target, tries=3):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Project C Colab)"})
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                target.write_bytes(response.read())
            return
        except Exception as error:
            print(f"  try {attempt} failed for {target.name}: {error}")
    raise RuntimeError(f"Could not download {target.name}. Open {url} in your browser, save the file, "
                       f"upload it to the Colab Files pane as {target.name}, and run this box again.")

for name, url in inputs.items():
    if not (root / name).exists():
        download(url, root / name)
    print(f"{name:20} {(root / name).stat().st_size:>12,} bytes")

## 2. Build the Census measures
### Box 3 · Poverty and vehicle rates for each tract
Poverty uses people for whom poverty status is determined (B17001: line 2 ÷ line 1). Vehicle availability uses households (B08201: households with no vehicle ÷ all households). Each rate divides the relevant count by its own denominator. A tract with a zero or missing denominator gets no rate.

In [ ]:
source = json.loads((root / "reporter.json").read_text())
rows = []
for geoid, tables in source["data"].items():
    row = {"tract": geoid.split("US")[1]}                 # "14000US24510010100" -> "24510010100"
    for label, table in [("poverty", "B17001"), ("no_vehicle", "B08201")]:
        est = tables[table]["estimate"]
        total, count = (est.get(f"{table}{i:03d}") for i in (1, 2))   # line 001 = total, 002 = count
        row[label] = count / total * 100 if total and total > 0 and count is not None and count >= 0 else None
    rows.append(row)
acs = pd.DataFrame(rows).set_index("tract")

display(acs.describe().round(2))

## 3. Assign bus stops to Baltimore City census tracts
### Box 4 · Point-in-tract join
The analysis includes a stop only when its published coordinate falls inside a single Baltimore City tract (county code 510). It includes the city stops on routes that cross the city boundary. Stops in Baltimore County are left out.

In [ ]:
# Read the Maryland tract shapefile straight out of the ZIP and keep Baltimore City only.
with zipfile.ZipFile(root / "tract.zip") as tract_zip:
    with tract_zip.open("tl_2024_24_tract.shp") as shp, tract_zip.open("tl_2024_24_tract.dbf") as dbf:
        tract_records = [(r.record["GEOID"], shape(r.shape.__geo_interface__))
                         for r in shapefile.Reader(shp=shp, dbf=dbf).iterShapeRecords()
                         if r.record["COUNTYFP"] == "510"]

# A spatial index makes the "which tract is this point in?" lookup fast.
tract_tree = STRtree([geom for _, geom in tract_records])
stops = json.loads((repo / "docs/data/school-commute/stops.json").read_text())
stop_tract = {}
for stop_id, stop in stops.items():
    hits = tract_tree.query(Point(stop["lon"], stop["lat"]), predicate="within")
    if len(hits) == 1:                                   # exactly one tract, or the stop is dropped
        stop_tract[stop_id] = tract_records[hits[0]][0]

print(f"{len(stop_tract):,} stop coordinates matched to Baltimore City tracts.")

## 4. Load and filter the scheduled stop events
### Box 5 · Read the raw records and keep the commute windows
One row represents a scheduled stop on a bus trip. The file patterns select the morning (`-am`) and afternoon (`-pm`) stop-event records. The scheduled clock time decides whether an event is in its window: morning is 6:00–9:30 a.m. and afternoon is 2:00–5:30 p.m., with the end times excluded.

In [ ]:
files = sorted([*(repo / "data/school-commute-raw").glob("*-am.csv.gz"),
                *(repo / "data/school-commute-raw").glob("*-pm.csv.gz")])
events = pd.concat((pd.read_csv(f, dtype={"stop_id": str}) for f in files), ignore_index=True)

clock = pd.to_datetime(events.scheduled_local)
minute = clock.dt.hour * 60 + clock.dt.minute             # minutes after midnight
starts = events.window.map({"am": 360, "pm": 840})        # 6:00 a.m. and 2:00 p.m.
ends = events.window.map({"am": 570, "pm": 1050})         # 9:30 a.m. and 5:30 p.m. (excluded)
events = events.loc[minute.ge(starts) & minute.lt(ends)].copy()
events["tract"] = events.stop_id.map(stop_tract)          # blank for stops outside the city

print(f"{len(files)} files, {len(events):,} scheduled stop events in the commute windows")

## 5. Define the outcome and check coverage
### Box 6 · Late-or-canceled flag and coverage table
A *classified* event has a usable timing classification or a classified cancellation. The outcome of interest is **canceled or more than seven minutes late** (`late`, `severely_late`, `canceled`). Missing or unclassified observations affect coverage and are excluded from the rate's denominator.

In [ ]:
events["disrupted"] = events.outcome.isin(["late", "severely_late", "canceled"])
events["classified"] = events.outcome.isin(["on_time", "early", "late", "severely_late", "canceled"])
events["monitored"] = events.outcome.ne("not_monitored")
city = events.dropna(subset=["tract"]).join(acs, on="tract")   # city stops + their tract's ACS rates

assert not city.empty and city["tract"].str.startswith("24510").all()
print(f"{city.stop_id.nunique():,} stops and {len(city):,} scheduled city stop events")
print(f"Study dates: {city.service_date.min()} through {city.service_date.max()}")
coverage = city.groupby("window").agg(
    scheduled=("stop_id", "size"),
    monitored=("monitored", "sum"),
    classified=("classified", "sum"),
)
coverage["monitored_pct"] = 100 * coverage.monitored / coverage.scheduled
coverage["classified_pct"] = 100 * coverage.classified / coverage.scheduled
assert (coverage.classified <= coverage.monitored).all()
display(coverage.round(2))

## 6. Set the tract comparison groups
### Box 7 · Quarter cutoffs
The lowest and highest quarters are defined separately for poverty and vehicle availability using all city tracts with a valid estimate. The cutoffs are based on tracts rather than the number of bus stops or events.

In [ ]:
definitions = {"poverty": "Poverty rate", "no_vehicle": "Households without a vehicle"}
quartile_cutoffs = acs[list(definitions)].quantile([.25, .75])
display(quartile_cutoffs.rename(index={.25: "Lower cutoff", .75: "Upper cutoff"}).round(2))

## 7. Calculate pooled disruption rates
### Box 8 · Rates for the lowest and highest quarters
Pool classified stop events within each tract group and time window. The event counts and coverage stay next to the percentages so the denominators are visible.

In [ ]:
def quarter_groups(metric):
    """Stop events in the lowest and highest quarter of tracts for one Census measure."""
    lower, upper = quartile_cutoffs[metric]
    return pd.concat([
        city.loc[city[metric].le(lower)].assign(group="Lower quarter"),
        city.loc[city[metric].ge(upper)].assign(group="Upper quarter"),
    ])

comparison = []
for metric, label in definitions.items():
    groups = quarter_groups(metric)
    for window, key in [("Overall", None), ("Morning", "am"), ("Afternoon", "pm")]:
        window_rows = groups if key is None else groups.loc[groups.window.eq(key)]
        for group, sample in window_rows.groupby("group"):
            known = sample.loc[sample.classified]
            comparison.append({
                "Measure": label, "Window": window, "Quartile": group,
                "Tracts": sample.tract.nunique(), "Scheduled": len(sample),
                "Classified": len(known), "Late or canceled": int(known.disrupted.sum()),
                "Rate (%)": known.disrupted.mean() * 100,
                "Monitored (%)": sample.monitored.mean() * 100,
                "Classified (%)": len(known) / len(sample) * 100,
            })

rates = pd.DataFrame(comparison)
display(rates.round(2))

## 8. Compare stops on the same route and date
### Box 9 · Matched gap
Busier, longer routes might run through some neighborhoods more than others. To take that out, compare high- and low-quarter stops **on the same route, date and commute window**, then average the gaps, weighting each matched cell by the smaller group's event count. A positive gap means the upper quarter had more late or canceled events.

In [ ]:
matched_rows = []
for metric, label in definitions.items():
    groups = quarter_groups(metric)
    for window, key in [("Overall", None), ("Morning", "am"), ("Afternoon", "pm")]:
        window_rows = groups if key is None else groups.loc[groups.window.eq(key)]
        known = window_rows.loc[window_rows.classified]
        strata = known.groupby(["route_id", "service_date", "window", "group"]).disrupted
        paired = strata.agg(["size", "mean"]).unstack("group").dropna()   # cells with both groups
        weights = paired["size"].min(axis=1)
        gaps = paired["mean"]["Upper quarter"] - paired["mean"]["Lower quarter"]
        matched_rows.append({
            "Measure": label, "Window": window, "Matched cells": len(paired),
            "Routes": paired.index.get_level_values("route_id").nunique(),
            "Matched gap (pp)": 100 * (gaps * weights).sum() / weights.sum(),
        })

matched = pd.DataFrame(matched_rows)
display(matched.round(2))

## 9. Assign stops to named neighborhoods
### Box 10 · Point-in-neighborhood join
Join stop coordinates to Baltimore City's official neighborhood boundaries for the interactive map. The tract tables above remain the evidence for the research claim. If a stop does not match exactly one neighborhood, its events stay in the citywide totals only. `make_valid` repairs a few city boundary shapes that cross themselves.

In [ ]:
features = json.loads((root / "neighborhoods.json").read_text())["features"]
features = [f for f in features if f["geometry"] and f["properties"]["Name"]]
hood_records = [(str(f["properties"]["OBJECTID"]), make_valid(shape(f["geometry"]))) for f in features]
names = {str(f["properties"]["OBJECTID"]): f["properties"]["Name"] for f in features}
hood_tree = STRtree([g for _, g in hood_records])

mapping = {}
for stop_id in city.stop_id.unique():
    stop = stops[stop_id]
    hits = hood_tree.query(Point(stop["lon"], stop["lat"]), predicate="covered_by")
    if len(hits) == 1:
        mapping[stop_id] = hood_records[hits[0]][0]
city = city.assign(neighborhood=city.stop_id.map(mapping))
print(f"{len(names)} neighborhoods. {city.neighborhood.isna().sum():,} city events remain in the citywide totals only.")

### Box 11 · Neighborhood service and demographic measures
`metrics()` computes every number shown on a page card: the late-or-canceled rate, on-time share, average and median minutes versus schedule, coverage, and the outcome counts. It runs for the whole city and each neighborhood, for Overall, Morning and Afternoon.

Neighborhood demographics are estimated from the area overlap between tracts and neighborhoods. This assumes residents and households are evenly spread within a tract, so the neighborhood estimates are approximate.

In [ ]:
def metrics(rows):
    """All page figures for one set of scheduled stop events."""
    known = rows.loc[rows.classified]                       # rate denominator
    timed = known.loc[known.outcome.ne("canceled")]         # events with a minute value
    delays = pd.to_numeric(timed.delay_seconds, errors="coerce").dropna() / 60
    watched = rows.loc[rows.monitored]
    return dict(n=len(known), monitored=len(watched), scheduled=len(rows),
                coverage=100*len(watched)/len(rows) if len(rows) else None,
                failed=100*known.disrupted.mean() if len(known) else None,
                on=100*known.outcome.eq("on_time").mean() if len(known) else None,
                cancel=int(known.outcome.eq("canceled").sum()),
                late=int(known.outcome.isin(["late","severely_late"]).sum()),
                early=int(known.outcome.eq("early").sum()),
                missing=int(watched.outcome.eq("no_realtime_record").sum()),
                skipped=int(watched.outcome.eq("skipped").sum()),
                unclassified=len(watched)-len(known)-int(watched.outcome.isin(["no_realtime_record","skipped"]).sum()),
                timed=len(delays), avg=float(delays.mean()) if len(delays) else None,
                median=float(delays.median()) if len(delays) else None)

# Service figures: "all" = both windows together, "am" = morning, "pm" = afternoon.
# (The page's code calls each neighborhood a "tract" internally; the keys are neighborhood ids.)
payload = {}
for window in ["all", "am", "pm"]:
    subset = city if window == "all" else city[city.window.eq(window)]
    payload[window] = {"city": metrics(subset),
                       "tracts": {str(t): metrics(rows) for t, rows in subset.groupby("neighborhood")}}
payload["names"] = names

# Demographics: share each tract's counts by how much of the tract lies in the neighborhood.
tract_shapes = [(geoid, make_valid(geom)) for geoid, geom in tract_records]
overlap_tree = STRtree([geom for _, geom in tract_shapes])
demographics = {}
for neighborhood, geom in hood_records:
    totals = {key: [0.0, 0.0] for key in ("poverty", "vehicles")}   # [count, denominator]
    for index in overlap_tree.query(geom, predicate="intersects"):
        geoid, tract_geom = tract_shapes[index]
        weight = geom.intersection(tract_geom).area / tract_geom.area
        if weight <= 0:
            continue
        tables = source["data"].get("14000US" + geoid, {})
        for key, table in [("poverty", "B17001"), ("vehicles", "B08201")]:
            estimates = tables.get(table, {}).get("estimate", {})
            total, count = (estimates.get(f"{table}{i:03d}") for i in (1, 2))
            if total and total > 0 and count is not None and 0 <= count <= total:
                totals[key][0] += count * weight
                totals[key][1] += total * weight
    demographics[neighborhood] = {key: 100 * count / total if total else None
                                  for key, (count, total) in totals.items()}
payload["demographics"] = demographics

### Box 12 · Export the data for the interactive map
The JSON holds the citywide and neighborhood results, demographic estimates, the date range, and the tract comparisons used in the page's claim (`equity`). The page reads everything it shows from this file.

In [ ]:
def comparisons(metric):
    """Lowest- vs highest-quarter rates and matched gaps, in the format the page reads."""
    lo, hi = acs[metric].quantile([.25, .75])
    groups = pd.concat([city.loc[city[metric].le(lo)].assign(group="low"),
                        city.loc[city[metric].ge(hi)].assign(group="high")])
    known = groups.loc[groups.classified]
    result = {}
    for window in ("all", "am", "pm"):
        sample = known if window == "all" else known.loc[known.window.eq(window)]
        group_rates = sample.groupby("group").disrupted.mean() * 100
        result[window] = {group: float(group_rates[group]) if group in group_rates else None
                          for group in ("low", "high")}
    for window in ("am", "pm"):
        sample = known.loc[known.window.eq(window)]
        paired = sample.groupby(["route_id", "service_date", "group"]).disrupted.agg(["size", "mean"]).unstack()
        if {"low", "high"}.issubset(paired["mean"].columns):
            paired = paired.dropna()
            weights = paired["size"].min(axis=1)
            diff = paired["mean"]["high"] - paired["mean"]["low"]
            result["matched_" + window] = float(100 * (diff * weights).sum() / weights.sum()) if weights.sum() else None
        else:
            result["matched_" + window] = None
    return result

payload["dates"] = {"first": str(city.service_date.min()), "last": str(city.service_date.max())}
payload["equity"] = {metric: comparisons(metric) for metric in ("poverty", "no_vehicle")}

data_json = json.dumps(payload, allow_nan=False, separators=(",", ":"))
Path("Baltimore_City_MTA_Map_Data.json").write_text(data_json)
print(f"Exported {len(names)} neighborhoods and {payload['all']['city']['n']:,} classified citywide events.")
for key, label in [("all", "Overall"), ("am", "Morning"), ("pm", "Afternoon")]:
    print(f"  {label:9}  {payload[key]['city']['failed']:.1f}% canceled or more than 7 minutes late")

## 10. Draw the graphic
### Box 13 · Turn the neighborhood boundaries into map shapes
The page draws the map as SVG, a plain-text drawing format. This box converts each neighborhood's longitude/latitude outline into screen coordinates. Longitude is shrunk by 0.775 (the cosine of Baltimore's latitude) so the city keeps its true shape, north is up, and there is a 20-pixel margin. Each shape carries its neighborhood id and name so the page can color it and show details on hover.

In [ ]:
from html import escape
from shapely.ops import unary_union

# Bounding box of the whole city, in longitude (x) and latitude (y).
xmin, ymin, xmax, ymax = unary_union([g for _, g in hood_records]).bounds
X_STRETCH = 0.775            # about cos(39.3 degrees): a degree of longitude is shorter than one of latitude
scale = min(560 / ((xmax - xmin) * X_STRETCH), 535 / (ymax - ymin))   # fit inside the drawing area
MAP_WIDTH = round((xmax - xmin) * scale * X_STRETCH + 40)             # plus a 20-pixel margin each side
MAP_HEIGHT = round((ymax - ymin) * scale + 40)

def project(lon, lat):
    """Longitude/latitude -> page pixels, with north at the top."""
    return 20 + (lon - xmin) * scale * X_STRETCH, 20 + (ymax - lat) * scale

def polygons_of(geom):
    """The separate polygons inside a shape (some neighborhoods have more than one piece)."""
    if geom.geom_type == "Polygon":
        yield geom
    elif hasattr(geom, "geoms"):
        for child in geom.geoms:
            yield from polygons_of(child)

def outline_rings(geom):
    """Every outline: each piece's outer edge, then any holes in it."""
    return [ring for p in polygons_of(geom) for ring in (p.exterior, *p.interiors)]

def svg_ring(ring):
    """SVG path text: M = move to the first corner, L = line to the next, Z = close."""
    return "M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in (project(*pt) for pt in ring.coords)) + "Z"

hood_shapes = dict(hood_records)
map_paths = "".join(
    f'<path d="{"".join(svg_ring(r) for r in outline_rings(geom))}" data-tract="{i}" '
    f'tabindex="0" role="button" aria-label="{escape(names[i], quote=True)}"/>'
    for i, geom in hood_records)
print(f"{len(hood_records)} neighborhood shapes on a {MAP_WIDTH} x {MAP_HEIGHT} map")

### Box 14 · Preview the maps in Colab
A quick picture of the page's map for each commute window, using the page's exact Maryland color scale: under 30%, 30–40%, 40–50%, 50–60% and 60% or more of classified stop events canceled or more than 7 minutes late. Gray means fewer than 20 classified events. Diagonal lines mark neighborhoods where 30% or more of people are below the poverty threshold, the same threshold as the page's striped overlay. The interactive version, with hover details, is at the end.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch, Patch
from matplotlib.path import Path as DrawPath

COLORS = ["#FFF1CC", "#FFB81C", "#EE7D8B", "#C8102E", "#111111"]   # Maryland: pale gold, gold, pink, red, black
GRAY, EDGES = "#d9d3d2", [30, 40, 50, 60]

def bin_color(value):
    return GRAY if value is None else COLORS[sum(value >= e for e in EDGES)]

def draw_shape(ax, geom, **style):
    vertices, codes = [], []
    for ring in outline_rings(geom):
        xy = [project(*pt) for pt in ring.coords]
        vertices += xy
        codes += [DrawPath.MOVETO] + [DrawPath.LINETO] * (len(xy) - 2) + [DrawPath.CLOSEPOLY]
    ax.add_patch(PathPatch(DrawPath(vertices, codes), **style))

fig, axes = plt.subplots(1, 3, figsize=(15, 6.6))
for ax, (key, label) in zip(axes, [("all", "Overall"), ("am", "Morning"), ("pm", "Afternoon")]):
    for i, geom in hood_records:
        m = payload[key]["tracts"].get(i)
        value = m["failed"] if m and m["n"] >= 20 else None      # same 20-event rule as the page
        draw_shape(ax, geom, facecolor=bin_color(value), edgecolor="white", linewidth=0.4)
        if (demographics[i]["poverty"] or 0) >= 30:
            draw_shape(ax, geom, facecolor="none", edgecolor="#21212b", linewidth=0, hatch="////")
    ax.set_xlim(0, MAP_WIDTH); ax.set_ylim(MAP_HEIGHT, 0); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"{label}: {payload[key]['city']['failed']:.1f}% citywide", fontsize=13)
legend = [Patch(color=c, label=t) for c, t in zip(COLORS + [GRAY],
          ["Under 30%", "30–<40%", "40–<50%", "50–<60%", "60% or more", "Fewer than 20 events"])]
legend.append(Patch(facecolor="white", edgecolor="#21212b", hatch="////", label="30%+ below poverty"))
fig.legend(handles=legend, loc="lower center", ncol=7, frameon=False)
fig.suptitle("Baltimore City bus stop events canceled or more than 7 minutes late, by neighborhood\n"
             f"School commute windows, {payload['dates']['first']} to {payload['dates']['last']}", fontsize=14)
fig.tight_layout(rect=(0, 0.06, 1, 0.94))
fig.savefig("Baltimore_City_MTA_Neighborhood_Maps.png", dpi=150)
plt.show()

### Box 15 · Chart the tract comparison
The bars show the claim's evidence: the share of classified stop events canceled or more than 7 minutes late in the lowest and highest quarters of tracts, for poverty and for households without a vehicle. The numbers over the morning bars are the same-route, same-date gaps from Box 9.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
windows = [("all", "Overall"), ("am", "Morning"), ("pm", "Afternoon")]
for ax, (metric, title, low_label, high_label) in zip(axes, [
        ("poverty", "Poverty rate", "Lowest-poverty quarter", "Highest-poverty quarter"),
        ("no_vehicle", "Households without a vehicle", "Fewest without a car", "Most without a car")]):
    eq = payload["equity"][metric]
    x = range(len(windows))
    low = [eq[k]["low"] for k, _ in windows]; high = [eq[k]["high"] for k, _ in windows]
    ax.bar([i - 0.2 for i in x], low, 0.4, color="#FFB81C", label=low_label)
    ax.bar([i + 0.2 for i in x], high, 0.4, color="#C8102E", label=high_label)
    for i, (a, b) in enumerate(zip(low, high)):
        ax.text(i - 0.2, a + 0.8, f"{a:.1f}%", ha="center", fontsize=9)
        ax.text(i + 0.2, b + 0.8, f"{b:.1f}%", ha="center", fontsize=9)
    ax.text(1, max(low[1], high[1]) + 6, f"same route & date: {eq['matched_am']:+.1f} pts",
            ha="center", fontsize=9, style="italic")
    ax.set_xticks(list(x), [label for _, label in windows]); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="upper left")
axes[0].set_ylabel("Canceled or more than 7 min late (%)"); axes[0].set_ylim(0, 70)
fig.suptitle("Bus service across Baltimore City tract groups")
fig.tight_layout()
fig.savefig("Baltimore_City_MTA_Tract_Comparison.png", dpi=150)
plt.show()

## 11. Build the web page
The next three boxes hold the page itself: its styling, its text, and the small JavaScript program that makes it interactive. They are written as Python text so the notebook can fill in the fresh data and map shapes. You do not need to edit them.

### Box 16 · Page styling (CSS)
The page uses the tracker site's look: navy, gold and brick on cream cards, with navy pill buttons. The map itself is shaded in Maryland flag colors, from pale gold through gold, pink and red to black.

In [ ]:
# The <head> and the CSS rules that style the page.
PAGE_HEAD = r'''<!doctype html>
<html lang="en">
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Baltimore City MTA Reliability Map | Project C</title>
<style>
'''

PAGE_CSS = r'''@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Libre+Baskerville:wght@400;700&display=swap');
/* Site look (same as the tracker home page): navy, gold and brick on cream cards. */
:root{color-scheme:light;--navy:#092644;--ink:#18314c;--gold:#f6bd42;--brick:#a44b3e;--paper:#fffaf2;--line:#eadfd1}
*{box-sizing:border-box}body{margin:0;background:linear-gradient(180deg,#fff6e7 0%,#fffdfa 55%,#eaf4f7 100%);color:var(--ink);font:16px/1.6 'DM Sans',system-ui,sans-serif}
header{background:linear-gradient(110deg,#fff1dc 0%,#ffd5b2 46%,#f5a48c 100%);color:var(--navy);border-bottom:7px solid var(--gold);padding:48px max(20px,calc((100% - 1050px)/2))}
header small{font-size:.8rem;text-transform:uppercase;letter-spacing:.15em;font-weight:800;color:#87351d}
header p{font-weight:600;color:#102c43}
h1,h2{font-family:'Libre Baskerville',Georgia,serif;line-height:1.2;color:var(--navy)}h1{font-size:clamp(2rem,5vw,3.6rem);letter-spacing:-.03em;margin:.3em 0}h3{color:var(--navy)}p{margin:.55em 0}
main{max-width:1090px;margin:auto;padding:28px 20px 56px}button,select{font:inherit}
button{color:var(--navy);padding:.45em 1.05em;border:2px solid var(--navy);background:#fff;border-radius:999px;font-weight:700;cursor:pointer}
button[aria-pressed=true]{background:var(--navy);color:#fff;border-color:var(--navy)}
button:focus-visible,select:focus-visible,a:focus-visible{outline:3px solid var(--gold);outline-offset:3px}
nav{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:18px}
article,aside,.map,main>section{background:#fffefa;border:1px solid var(--line);border-radius:18px;padding:22px;box-shadow:0 8px 26px rgba(8,37,65,.05);margin-bottom:18px}
article h2,aside h2,.map h2,main>section h2{margin-top:0}.metrics{display:grid;grid-template-columns:repeat(auto-fit,minmax(135px,1fr));gap:14px;margin:0}
.metrics div{border-left:3px solid var(--gold);padding-left:9px}.metrics dt{font-size:.85rem;color:#3f5167}.metrics dd{margin:0;font:700 1.4rem/1.3 'Libre Baskerville',Georgia,serif;color:var(--navy)}
.grid{display:grid;grid-template-columns:1fr;gap:16px;margin:16px 0;align-items:start}
svg{display:block;width:100%;max-height:580px;background:#f6f2ea;border-radius:10px}
path{stroke:#fff;stroke-width:.65;fill-rule:evenodd;cursor:pointer;vector-effect:non-scaling-stroke}
path:hover,path:focus,path.selected{stroke:var(--navy);stroke-width:2;outline:none}
.legend{display:flex;flex-wrap:wrap;gap:8px 14px;font-size:.8rem;margin-top:12px}.swatch{display:inline-block;width:13px;height:13px;border:1px solid #9a9a9a;border-radius:3px;vertical-align:middle;margin-right:4px}
.counts{padding-top:8px;font-size:.91rem}small{display:block}a{color:#155e87}select{max-width:100%;margin-top:8px;padding:.35em .5em;border:1px solid #c9ccd0;border-radius:8px}
.maps{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,245px),1fr));gap:14px}
.map-panel{min-width:0}.map-panel h3{margin:0 0 6px}.map-controls{margin-bottom:14px}
.comparison-scroll{overflow:auto}.comparison-scroll table{width:100%;border-collapse:collapse;font-size:.92rem}
.comparison-scroll th,.comparison-scroll td{text-align:left;padding:.6rem;border-bottom:1px solid var(--line)}
.comparison-scroll thead th{background:#f3efe4;color:var(--navy)}
path.comparison-hover{stroke:var(--navy);stroke-width:2}
.maps.combined{grid-template-columns:1fr}
.overlay-stripes{background-image:repeating-linear-gradient(135deg,transparent 0 4px,#21212b 4px 5px,white 5px 6px)}
.overlay-dots{background-image:radial-gradient(circle,#21212b 0 1px,white 1.2px 2px,transparent 2.1px);background-size:6px 6px}
#clear-selected{color:#fff;background:var(--brick);border-color:var(--brick);-webkit-text-fill-color:#fff}
.colab-link{display:inline-block;padding:10px 18px;border-radius:8px;background:var(--navy);color:#fff;font-weight:600;text-decoration:none}.colab-link:hover{opacity:.9}
'''

### Box 17 · Page text
The header, claim, citywide cards, map controls, and the sections on the law, what reliability means for students, and how to read the numbers. Numbers are not typed here: the script in Box 18 fills them in from the data.

In [ ]:
# Everything between </style> and the map drawing.
PAGE_BODY = r'''<header><small>PROJECT C · DATA SCIENCE FOR LAWYERS</small><h1>Baltimore City bus reliability</h1><p>Scheduled MTA local bus stop events · <span id="service-dates"></span></p></header>
<main>
<article aria-labelledby="project-claim" style="margin-bottom:18px;border-left:5px solid var(--gold)">
<h2 id="project-claim">Project C claim</h2>
<p><strong id="current-claim"></strong></p>
<p>Maryland law requires MTA to provide fare-free transit for eligible Baltimore City public school students for covered activities. The project asks how dependable that service is during school commute hours. The figures describe stop events across the public bus network. They do not measure the share of eligible students whose school trips were disrupted. <a href="https://mgaleg.maryland.gov/2026RS/Statute_Web/gtr/7-710.pdf">Md. Code Ann., Transp. § 7–710(a)–(c)</a></p>
<p id="equity-summary"></p>

<p>The comparison uses Census tracts to measure poverty and vehicle access. The map below reports service by named neighborhood. Household vehicle access is an indicator of transportation options. The records cannot identify which students rode the bus or whether they arrived at school late.</p>
<p><a href="https://github.com/TbilKitty/MTA_Accountability_Tracker/tree/main/data/school-commute-raw">MTA stop-event records</a> · <a href="https://data.census.gov/table/ACSDT5Y2024.B17001">2020–2024 ACS poverty estimates</a> · <a href="https://data.census.gov/table/ACSDT5Y2024.B08201">2020–2024 ACS vehicle availability</a> · <a href="https://catalog.data.gov/dataset/tiger-line-shapefile-2024-state-maryland-md-census-tract">Census tract boundaries</a></p>
<p><a href="https://colab.research.google.com/drive/1O3dTTWmpzxq53g5wvCWuFN4wT2QTOQKO" class="colab-link">Open the analysis in Colab</a></p>
</article>
<nav aria-label="Commute window"><button data-window="all" aria-pressed="true">Overall</button><button data-window="am" aria-pressed="false">Morning</button><button data-window="pm" aria-pressed="false">Afternoon</button></nav>
<article><h2 id="city-title">Citywide service</h2>
<p>Each record is one scheduled stop on one bus trip within Baltimore City. A trip can contribute several stop events.</p>
<div id="city"></div>
<details><summary>How to read the citywide numbers</summary>
<dl>
<dt><strong>Canceled or more than 7 minutes late</strong></dt><dd>The share of classified stop events with either outcome. A cancellation counts when the real-time feed flagged the trip as canceled and the tracker found no matching vehicle position.</dd>
<dt><strong>On time among classified events</strong></dt><dd>The share within two minutes early through seven minutes late. The denominator includes all timing classifications and classified cancellations.</dd>
<dt><strong>Average vs. schedule</strong></dt><dd>Add up the estimated timing differences and divide by the number of events with usable timing estimates. For example, an average of 4 min late means the measured stop events were four minutes behind schedule on average. Early times reduce the average. Cancellations and missing timings have no minute value and are excluded. The measure describes schedule adherence. Rider wait times are unknown.</dd>
<dt><strong>Median vs. schedule</strong></dt><dd>The middle timing difference when measured events are ordered from earliest to latest. It is less affected by a few unusually long delays.</dd>
<dt><strong>Monitoring coverage</strong></dt><dd>The share of scheduled stop events that fell within the tracker’s monitoring coverage. Some monitored events still lack a usable timing classification.</dd>
<dt><strong>Classified events</strong></dt><dd>The count of stop events classified as early, on time, late, very late, or canceled. Missing vehicle records, feed-flagged skips, and unclassified timing are shown separately in the counts below the cards.</dd>
</dl>
</details></article>
<div class="grid">
<section class="map"><h2>Baltimore by neighborhood</h2><p>Combine measures on one map or compare them side by side. Click neighborhoods to add or remove them from the comparison.</p>
<nav aria-label="Map layout"><button data-layout="combined" aria-pressed="true">Combined map</button><button data-layout="separate" aria-pressed="false">Side by side</button></nav>
<nav aria-label="Map shading"><button data-layer="service" aria-pressed="true">Bus reliability</button><button data-layer="vehicles" aria-pressed="true">Vehicle access</button><button data-layer="poverty" aria-pressed="true">Poverty</button></nav>
<div class="map-controls"><label for="tract-select">Add a neighborhood</label> <select id="tract-select"><option value="">Select a neighborhood</option></select> <button id="clear-selected" type="button">Clear selected neighborhoods</button></div>
'''

# Everything after the map, up to the embedded data.
PAGE_AFTER_MAP = r'''
<div class="maps combined" id="maps">@@STATIC_MAP@@</div><p id="overlay-note" class="counts"></p></section>
<section class="map" aria-labelledby="compare-title"><h2 id="compare-title">Selected neighborhoods</h2><div class="comparison-scroll" id="comparison">Select neighborhoods on any map to compare their figures.</div></section>
<aside aria-label="Neighborhood details"><h2 id="tract-title">Neighborhood details</h2><div id="tract">Choose a neighborhood on the map or from the list.</div></aside>
</div>
<section><h2>Bus service across poverty groups</h2>
<p id="poverty-summary"></p>
<div class="comparison-scroll"><table>
<caption>Share canceled or more than seven minutes late among classified stop events</caption>
<thead><tr><th scope="col">Commute window</th><th scope="col">Lowest-poverty quarter</th><th scope="col">Highest-poverty quarter</th></tr></thead>
<tbody id="poverty-rates"></tbody></table></div>
<p>Tract poverty rates describe the area around a stop. Riders’ incomes are unknown.</p>
<p><a href="https://github.com/TbilKitty/MTA_Accountability_Tracker/tree/main/data/school-commute-raw">MTA stop-event records</a> · <a href="https://data.census.gov/table/ACSDT5Y2024.B17001">2020–2024 ACS poverty estimates</a></p></section>
<section class="map" aria-labelledby="school-access" style="margin-top:18px">
<h2 id="school-access">What bus reliability means for students</h2>
<p>Baltimore City Public Schools and MTA provide eligible students free public transit, including local bus service. For a student who depends on that bus, an early departure can mean waiting for the next one. A late or canceled bus can cause a missed connection and leave too little time to get to class. That risk matters especially when a student has few alternatives. The tracker measures service at stops during school commute hours. It does not record who boarded, whether students missed connections, or how much class time they lost.</p>
<p><a href="https://www.baltimorecityschools.org/article/3080724">City Schools’ 2026–27 student transit program</a> · <a href="https://www.mta.maryland.gov/student-fares">MTA student fares and eligibility</a> · <a href="https://www.thebanner.com/education/k-12-schools/baltimore-city-students-mta-EN4OO6SETNHUHGQETMHGKWTFZA/">The Baltimore Banner’s school-commute analysis</a></p>

<h3>What MTA and City Schools are responsible for</h3>
<p>Maryland’s Transportation Article, § 7–710(a)–(c), requires MTA to provide fare-free transit to eligible Baltimore City public school students for school-related or educational extracurricular activities between 5 a.m. and 8 p.m. Under subsection (d), MTA must work with Baltimore City Public Schools and the Mayor’s Office of Employment Development to adopt eligibility regulations for the student and YouthWorks transit-pass programs. Maryland’s student transportation regulations separately make the local school system responsible for the safe operation of its student transportation system.</p>
<p>Section 7–710 requires access to covered transit from 5 a.m. to 8 p.m. It sets no punctuality, trip-completion, or arrival-at-school benchmark. The tracker samples the 6–9:30 a.m. and 2–5:30 p.m. commute windows. It cannot determine how consistently MTA provided access throughout the full statutory period.</p>
<p>The Baltimore Regional Transit Commission’s 2025 annual report describes an agreement under which MTA provides adjusted public bus service to accommodate students. The tracker’s citywide figures include general local bus stops and do not isolate service covered by that agreement. Assessing compliance would require the agreement and applicable rules, service records for affected routes, and evidence of what students experienced.</p>
<p><a href="https://mgaleg.maryland.gov/2026RS/Statute_Web/gtr/7-710.pdf">Md. Code Ann., Transp. § 7–710</a> · <a href="https://regs.maryland.gov/us/md/exec/comar/11.06.05">COMAR 11.06.05, Baltimore City student transit</a> · <a href="https://regs.maryland.gov/us/md/exec/comar/13A.06.07.03">COMAR 13A.06.07.03, student transportation</a> · <a href="https://dlslibrary.state.md.us/publications/ICA/BMC/TR7-213%28i%29_2025.pdf">Baltimore Regional Transit Commission, 2025 annual report</a></p>

<h3>What should count as reliable for a school trip?</h3>
<p>MTA counts a local bus as on time within a window of two minutes early to seven minutes late. A bus can meet that benchmark and still arrive too late for a student’s transfer or first bell. The tracker uses the same tolerance for its own stop-event estimates. The citywide cards describe that sample and use different coverage and classification rules from MTA’s published performance series.</p>
<div class="comparison-scroll"><table>
<thead><tr><th scope="col">Measure</th><th scope="col">What it would tell us</th></tr></thead>
<tbody>
<tr><th scope="row">On-time stop events</th><td>How often service falls within the schedule tolerance. Report coverage alongside the rate so readers can see how much service was observed.</td></tr>
<tr><th scope="row">Cancellations and early departures</th><td>How often a planned boarding opportunity may be lost. Track confirmed cancellations and early departures separately from missing vehicle records.</td></tr>
<tr><th scope="row">Long waits and missed connections</th><td>How often students have to wait much longer than planned, and how much time they need to leave for transfers. Measuring this requires trip-level data.</td></tr>
<tr><th scope="row">Arrival before the first bell</th><td>What share of observed school trips arrive before the school’s actual start time, and how many minutes of instruction are missed when they do not. Measuring this requires student-trip or attendance evidence.</td></tr>
</tbody></table></div>
<h3>What MTA and City Schools could report</h3>
<ul>
<li><strong>MTA service delivery.</strong> On routes and trips that serve schools, report how many scheduled trips operated, how many were canceled, and how often buses left key boarding stops early. Show missing records separately and include the number of scheduled trips behind each rate.</li>
<li><strong>Student access to instruction.</strong> City Schools and MTA could track how often sampled school journeys reach the school before the bell, how much extra time students need for a dependable trip, and which disruptions cause missed connections or class time. Report results by school and route using aggregate records.</li>
</ul>
<p>These are proposed accountability measures. MTA’s schedule tolerance helps assess bus operations. Arrival before the bell tells us whether the full trip worked for students.</p>
<p><a href="https://www.mta.maryland.gov/performance-improvement">MTA on-time performance definition</a> · <a href="https://github.com/TbilKitty/MTA_Accountability_Tracker/tree/main/data/school-commute-raw">Project C stop-event records</a></p>
</section>
<section><h2>How to read the numbers</h2><p>The map counts stops inside Baltimore City, including city stops on routes that cross the city line. Classified events include timing estimates and eligible feed-flagged cancellations. Missing vehicle records can reflect a missed trip or a tracking gap. A single bus trip can contribute several stop events.</p>
<p>Citywide totals include city stops that could not be assigned to a single neighborhood.</p>
<p><strong>Neighborhood demographics.</strong> The 2020–2024 American Community Survey estimates are fixed across commute views. Neighborhood figures are approximate: we allocate each tract’s counts by the share of its area overlapping the neighborhood, sum the estimated counts, and divide by the estimated population or household total. This assumes people and households are evenly distributed within each tract. The project’s claim was tested using the original Census tract rates.</p>
<p><a href="https://github.com/TbilKitty/MTA_Accountability_Tracker/tree/main/data/school-commute-raw">MTA stop-event records</a> · <a href="https://baltegis.baltimorecity.gov/server/rest/services/CityView/Neighborhoods/FeatureServer">Baltimore City neighborhood boundaries</a></p></section>
</main>
'''

### Box 18 · Page script (JavaScript)
Reads the embedded data, writes the claim sentence and tables, colors the map, and handles hovering, clicking and the Overall/Morning/Afternoon buttons.

In [ ]:
PAGE_SCRIPT = r'''const data = JSON.parse(document.getElementById('data').textContent);
const maps = document.getElementById('maps');
// The page ships with the overall map already drawn, so it shows even where scripts
// don't run (file previews). The script copies that drawing for each interactive map.
const mapSource = maps.querySelector('svg');
const select = document.getElementById('tract-select');
const colors = ['#FFF1CC','#FFB81C','#EE7D8B','#C8102E','#111111'];
const layers = {
  service: {label:'classified stop events canceled or more than 7 minutes late', edges:[30,40,50,60], missing:'Fewer than 20 classified events or no data'},
  vehicles: {label:'households without a vehicle (approximate)', edges:[10,20,30,40], missing:'No estimate'},
  poverty: {label:'people below the poverty threshold (approximate)', edges:[10,20,30,40], missing:'No estimate'}
};
const activeLayers = new Set(['service', 'vehicles', 'poverty']);
let layout = 'combined';
const overlays = {
  poverty: {pattern:'stripes', threshold:30, label:'Stripes · 30% or more of people below poverty threshold'},
  vehicles: {pattern:'dots', threshold:40, label:'Dots · 40% or more of households without a vehicle'}
};
const layerNames = {service:'Bus reliability', vehicles:'Households without a vehicle', poverty:'People below the poverty threshold'};
const gray = '#d9d3d2';
const number = new Intl.NumberFormat('en-US');
const pct = value => value == null ? '—' : value.toFixed(1) + '%';
const timing = value => {
  if (value == null) return '—';
  const minutes = Math.abs(value).toFixed(1);
  return minutes === '0.0' ? '0.0 min' : minutes + (value > 0 ? ' min late' : ' min early');
};
const tractName = id => data.names[id];
const dateLabel = iso => new Intl.DateTimeFormat('en-US', {
  month:'long', day:'numeric', year:'numeric', timeZone:'UTC'
}).format(new Date(iso + 'T00:00:00Z'));
document.getElementById('service-dates').textContent =
  dateLabel(data.dates.first) + ' – ' + dateLabel(data.dates.last);
const morningRate = data.am.city.failed;
document.getElementById('current-claim').textContent = morningRate == null ?
  'There are no classified morning stop events in this update.' :
  'During the morning school commute, ' + pct(morningRate) +
  ' of classified MTA local bus stop events in Baltimore City were canceled or more than seven minutes late.';
const {poverty, no_vehicle: vehicles} = data.equity;
const gap = value => value == null ? 'unavailable' : (value > 0 ? '+' : '') + value.toFixed(1) + ' points';
document.getElementById('equity-summary').textContent =
  'Among classified morning events, the late-or-canceled rate was ' + pct(poverty.am.high) +
  ' in the highest-poverty quarter and ' + pct(poverty.am.low) +
  ' in the lowest. For the highest and lowest quarters by households without a vehicle, the rates were ' +
  pct(vehicles.am.high) + ' and ' + pct(vehicles.am.low) +
  '. Comparing the same route and school date, the higher-minus-lower morning gaps were ' +
  gap(poverty.matched_am) + ' for poverty and ' + gap(vehicles.matched_am) + ' for vehicle access.';
const direction = ({high, low}) => high == null || low == null ? 'insufficient data' :
   pct(high) === pct(low) ? 'similar rates in both groups' :
   high > low ? 'a higher rate in the highest-poverty quarter' : 'a higher rate in the lowest-poverty quarter';
document.getElementById('poverty-summary').textContent =
  'The morning comparison showed ' + direction(poverty.am) +
  '. The afternoon comparison showed ' + direction(poverty.pm) + '.';
const povertyBody = document.getElementById('poverty-rates');
for (const [key, label] of [['all','Overall'], ['am','Morning'], ['pm','Afternoon']]) {
  const row = povertyBody.insertRow();
  const heading = document.createElement('th');
  heading.scope = 'row';
  heading.textContent = label;
  row.append(heading);
  row.insertCell().textContent = pct(poverty[key].low);
  row.insertCell().textContent = pct(poverty[key].high);
}
let windowKey = 'all';
const selected = new Set();
const panels = new Map();
let focused = '';
Object.keys(data.names).sort((a,b)=>tractName(a).localeCompare(tractName(b)))
  .forEach(id=>select.add(new Option(tractName(id),id)));
function metrics(m) {
  if (!m) return '<p>No scheduled stop events in this neighborhood and window.</p>';
  const values = [['Canceled or >7 min late',pct(m.failed)],['On time among classified',pct(m.on)],
    ['Average vs. schedule',timing(m.avg)],['Median vs. schedule',timing(m.median)],
    ['Monitoring coverage',pct(m.coverage)],['Classified events',number.format(m.n)]];
  return '<dl class="metrics">'+values.map(([label,value])=>'<div><dt>'+label+'</dt><dd>'+value+'</dd></div>').join('')+'</dl>'+
    '<p class="counts">'+number.format(m.cancel)+' canceled · '+number.format(m.late)+' >7 min late · '+
    number.format(m.early)+' >2 min early by estimated timing · '+number.format(m.skipped)+' skipped · '+
    number.format(m.missing)+' missing vehicle records · '+number.format(m.unclassified)+' with unclassified timing</p>'+
    '<small>Timing averages use '+number.format(m.timed)+' events. Coverage uses '+number.format(m.monitored)+' monitored of '+number.format(m.scheduled)+' scheduled events.</small>';
}
function tract(id) {
  const title = document.getElementById('tract-title'), panel = document.getElementById('tract');
  title.textContent = id ? tractName(id) : 'Neighborhood details';
  if (!id) {panel.textContent='Choose a neighborhood on the map or from the list.'; return;}
  const m = data[windowKey].tracts[id];
  const {poverty, vehicles} = data.demographics[id] || {};
  const withVehicle = vehicles == null ? null : 100 - vehicles;
  const context = '<h3>Neighborhood demographics</h3><dl class="metrics">'+
    [['Households without a vehicle', vehicles], ['Households with a vehicle', withVehicle],
     ['People below the poverty threshold', poverty]]
    .map(([label,value])=>'<div><dt>'+label+'</dt><dd>'+pct(value)+'</dd></div>').join('')+
    '</dl><p><small>Approximate neighborhood estimates from the 2020–2024 ACS. These figures stay the same across commute windows.</small></p>';
  panel.innerHTML = context + '<h3>Bus service</h3>' + metrics(m) + (m && m.n < 20 ? '<p><small>Small sample. Rate shown for context and omitted from map shading.</small></p>' : '');
}
function renderSelection() {
  for (const panel of panels.values()) for (const path of panel.paths) {
    const id = path.dataset.tract;
    path.classList.toggle('selected', selected.has(id));
    path.classList.toggle('comparison-hover', focused === id);
    path.setAttribute('aria-pressed', String(selected.has(id)));
  }
}
function comparison() {
  const target = document.getElementById('comparison');
  target.replaceChildren();
  if (!selected.size) {
    target.textContent = 'Select neighborhoods on any map to compare their figures.';
    return;
  }
  const table = document.createElement('table');
  const head = table.createTHead().insertRow();
  for (const label of ['Neighborhood', 'Households without a vehicle', 'People below poverty threshold',
                       'Canceled or >7 min late', 'Classified stop events', '']) {
    const th = document.createElement('th');
    th.scope = 'col';
    th.textContent = label;
    head.append(th);
  }
  const body = table.createTBody();
  for (const id of selected) {
    const m = data[windowKey].tracts[id], d = data.demographics[id];
    const row = body.insertRow();
    for (const value of [tractName(id), pct(d?.vehicles), pct(d?.poverty),
                         pct(m?.failed), m ? number.format(m.n) : '—']) row.insertCell().textContent = value;
    const remove = document.createElement('button');
    remove.textContent = 'Remove';
    remove.setAttribute('aria-label', 'Remove ' + tractName(id));
    remove.addEventListener('click', () => choose(id));
    row.insertCell().append(remove);
  }
  target.append(table);
}
function choose(id) {
  if (!id) return;
  if (selected.has(id)) selected.delete(id);
  else selected.add(id);
  focused = id;
  renderSelection();
  comparison();
  tract(id);
}
function preview(id) {
  focused = id;
  renderSelection();
  tract(id);
}
function buildMaps() {
  maps.replaceChildren();
  maps.classList.toggle('combined', layout === 'combined');
  document.getElementById('overlay-note').textContent = layout === 'combined' &&
    (activeLayers.has('vehicles') || activeLayers.has('poverty'))
    ? 'Patterns mark thresholds using approximate neighborhood estimates. Click a neighborhood for exact displayed percentages and data availability.'
    : '';
  const visible = layout === 'combined' ? ['combined'] : Object.keys(layers).filter(key => activeLayers.has(key));
  for (const layer of visible) {
    if (!panels.has(layer)) {
      const node = document.createElement('section');
      node.className = 'map-panel';
      const title = document.createElement('h3');
      title.textContent = layerNames[layer] || 'Combined neighborhood map';
      const svg = mapSource.cloneNode(true);
      svg.setAttribute('aria-label', title.textContent + ' by Baltimore neighborhood');
      const legend = document.createElement('div');
      legend.className = 'legend';
      node.append(title, svg, legend);
      const paths = [...svg.querySelectorAll('[data-tract]')];
      const patterns = {};
      if (layer === 'combined') {
        const ns = 'http://www.w3.org/2000/svg';
        const defs = document.createElementNS(ns, 'defs');
        defs.innerHTML = '<pattern id="overlay-stripes" width="9" height="9" patternUnits="userSpaceOnUse" patternTransform="rotate(45)"><path d="M2 0V9" style="stroke:white;stroke-width:2.8"/><path d="M2 0V9" style="stroke:#21212b;stroke-width:1.15"/></pattern>' +
          '<pattern id="overlay-dots" width="11" height="11" patternUnits="userSpaceOnUse"><circle cx="5.5" cy="5.5" r="2" fill="#21212b" stroke="white" stroke-width="0.9"/></pattern>';
        svg.prepend(defs);
        for (const [key, overlay] of Object.entries(overlays)) {
          const group = document.createElementNS(ns, 'g');
          group.setAttribute('aria-hidden', 'true');
          group.style.pointerEvents = 'none';
          for (const path of paths) {
            const value = data.demographics[path.dataset.tract]?.[key];
            if (value == null || value < overlay.threshold) continue;
            const mark = document.createElementNS(ns, 'path');
            mark.setAttribute('d', path.getAttribute('d'));
            mark.style.fill = 'url(#overlay-' + overlay.pattern + ')';
            mark.style.stroke = 'none';
            group.append(mark);
          }
          svg.append(group);
          patterns[key] = group;
        }
      }
      for (const path of paths) {
        const id = path.dataset.tract;
        path.addEventListener('mouseenter',()=>preview(id));
        path.addEventListener('mouseleave',()=>preview([...selected].at(-1) || ''));
        path.addEventListener('focus',()=>preview(id));
        path.addEventListener('click',()=>choose(id));
        path.addEventListener('keydown',e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();choose(id);}});
      }
      panels.set(layer, {node, paths, legend, patterns});
    }
    maps.append(panels.get(layer).node);
  }
}
function redraw() {
  document.getElementById('city-title').textContent = 'Citywide service · ' + {all:'Overall', am:'Morning', pm:'Afternoon'}[windowKey];
  document.getElementById('city').innerHTML = metrics(data[windowKey].city);
  for (const [layer, panel] of panels) {
    const combined = layer === 'combined';
    const showColor = !combined || activeLayers.has('service');
    const measure = combined ? 'service' : layer;
    const {edges, missing} = layers[measure];
    const bins = ['Under '+edges[0]+'%', ...edges.slice(1).map((n,i)=>edges[i]+'–<'+n+'%'),
                  edges.at(-1)+'% or more', missing];
    panel.legend.innerHTML = showColor ? (combined ? '<span><strong>Color · canceled or >7 min late</strong></span>' : '') +
      bins.map((text,i)=>'<span><i class="swatch" style="background:'+(colors[i]||gray)+'"></i>'+text+'</span>').join('') : '';
    for (const [key, group] of Object.entries(panel.patterns)) {
      group.style.display = activeLayers.has(key) ? '' : 'none';
      const overlay = overlays[key];
      if (activeLayers.has(key)) panel.legend.insertAdjacentHTML('beforeend',
        '<span><i class="swatch overlay-'+overlay.pattern+'"></i>'+overlay.label+'</span>');
    }
    for (const path of panel.paths) {
      const id = path.dataset.tract;
      const m = data[windowKey].tracts[id];
      const value = measure === 'service' ? (m?.n >= 20 ? m.failed : null) : data.demographics[id]?.[measure];
      path.style.fill = !showColor ? '#f1ece5' : value == null ? gray : colors[edges.filter(edge=>value>=edge).length];
    }
  }
  renderSelection();
  comparison();
  tract(focused);
}
document.querySelectorAll('[data-layout]').forEach(button=>button.addEventListener('click',()=>{
  layout = button.dataset.layout;
  document.querySelectorAll('[data-layout]').forEach(b=>b.setAttribute('aria-pressed',String(b===button)));
  buildMaps();
  redraw();
}));
document.querySelectorAll('[data-layer]').forEach(button=>button.addEventListener('click',()=>{
  const layer = button.dataset.layer;
  if (activeLayers.has(layer) && activeLayers.size === 1) return;
  if (activeLayers.has(layer)) activeLayers.delete(layer);
  else activeLayers.add(layer);
  button.setAttribute('aria-pressed',String(activeLayers.has(layer)));
  buildMaps();
  redraw();
}));
select.addEventListener('change',()=>{
  if (selected.has(select.value)) preview(select.value);
  else choose(select.value);
  select.value = '';
});
document.getElementById('clear-selected').addEventListener('click',()=>{
  selected.clear();
  preview('');
  comparison();
});
document.querySelectorAll('[data-window]').forEach(button=>button.addEventListener('click',()=>{
  windowKey=button.dataset.window;
  document.querySelectorAll('[data-window]').forEach(b=>b.setAttribute('aria-pressed',String(b===button)));
  redraw();
}));
buildMaps();
redraw();
'''

### Box 19 · Assemble and save the finished page
Puts the pieces together: styling, text, the map (already colored for the Overall view, so it shows even in file previews that don't run scripts), the JSON from Box 12, and the script. The result is one self-contained HTML file you can open in any browser or publish on GitHub Pages. Colab's Files pane (folder icon on the left) will show it with the JSON and the two PNG charts; right-click to download.

In [ ]:
safe_json = data_json.replace("</", "<\\/")      # keeps the data from closing its <script> tag early
import re

# Draw the overall map in advance (same colors as the page's script), so the map is
# visible even in a viewer that doesn't run scripts.
SERVICE_COLORS, SERVICE_EDGES = ["#FFF1CC", "#FFB81C", "#EE7D8B", "#C8102E", "#111111"], [30, 40, 50, 60]
def fill(neighborhood_id):
    m = payload["all"]["tracts"].get(neighborhood_id)
    if not m or m["n"] < 20:
        return "#d9d3d2"
    return SERVICE_COLORS[sum(m["failed"] >= edge for edge in SERVICE_EDGES)]
filled_paths = re.sub(r'data-tract="(\d+)"',
                      lambda hit: f'{hit.group(0)} style="fill:{fill(hit.group(1))}"', map_paths)
static_map = ('<section class="map-panel"><h3>Combined neighborhood map</h3>'
              f'<svg viewBox="0 0 {MAP_WIDTH} {MAP_HEIGHT}" role="group">{filled_paths}</svg>'
              '<noscript><p class="counts">Overall share canceled or more than 7 minutes late. '
              'Open the page in a web browser for the interactive map.</p></noscript></section>')

page = (PAGE_HEAD + PAGE_CSS + "</style>\n" + PAGE_BODY
        + PAGE_AFTER_MAP.replace("@@STATIC_MAP@@", static_map)
        + '<script id="data" type="application/json">' + safe_json + "</script>\n"
        + "<script>\n" + PAGE_SCRIPT + "</script>\n</html>\n")
Path("Baltimore_City_MTA_Reliability_Map.html").write_text(page)
print(f"Wrote Baltimore_City_MTA_Reliability_Map.html ({len(page)/1e6:.1f} MB)")

### Box 20 · Check that the numbers match the published page
The published page's numbers are written below. (With `USE_LATEST = True` they will differ, because there is more data.) This box compares them with what the notebook just calculated, and also compares a fingerprint (SHA-256 hash) of the whole data file, which matches only if every number on the page is identical.

In [ ]:
import hashlib
published = {   # from Baltimore_City_MTA_Reliability_Map.html, Oct 30 - Nov 20, 2026
    "Overall classified events": 128997, "Overall % canceled or >7 min late": 41.2165,
    "Morning % canceled or >7 min late": 32.0297, "Afternoon % canceled or >7 min late": 50.6204,
    "Morning, highest-poverty quarter %": 34.0088, "Morning, lowest-poverty quarter %": 27.9657,
    "Morning, most households without a car %": 35.5769, "Morning, fewest without a car %": 27.9292,
    "Morning matched gap, poverty (pts)": 3.1065, "Morning matched gap, vehicles (pts)": 3.7264,
}
eq = payload["equity"]
ours = {
    "Overall classified events": payload["all"]["city"]["n"],
    "Overall % canceled or >7 min late": payload["all"]["city"]["failed"],
    "Morning % canceled or >7 min late": payload["am"]["city"]["failed"],
    "Afternoon % canceled or >7 min late": payload["pm"]["city"]["failed"],
    "Morning, highest-poverty quarter %": eq["poverty"]["am"]["high"],
    "Morning, lowest-poverty quarter %": eq["poverty"]["am"]["low"],
    "Morning, most households without a car %": eq["no_vehicle"]["am"]["high"],
    "Morning, fewest without a car %": eq["no_vehicle"]["am"]["low"],
    "Morning matched gap, poverty (pts)": eq["poverty"]["matched_am"],
    "Morning matched gap, vehicles (pts)": eq["no_vehicle"]["matched_am"],
}
check = pd.DataFrame({"Published page": published, "This notebook": ours})
check["Match"] = (check["Published page"] - check["This notebook"]).abs() < 0.0001
display(check)
fingerprint = hashlib.sha256(data_json.encode()).hexdigest()
same_file = fingerprint == "339250a00228b293d20c04524ada897a8f205d95c14a4e146cbcc7f2e26368fc"
print("Whole data file identical to the published page:", same_file)

### Box 21 · Show the interactive page here
This is the finished page running inside Colab. Hover over or click a neighborhood, and use the Overall / Morning / Afternoon buttons.

In [ ]:
display(HTML(page))

## 12. What the evidence supports
The figures below are from the pinned October 30–November 20, 2026 sample (Boxes 8, 9 and 12).

Among Baltimore City stops studied, the morning share of classified events canceled or more than seven minutes late was **34.0% in the highest-poverty tracts versus 28.0% in the lowest-poverty tracts**, and **35.6% versus 27.9%** in the highest and lowest quarters of households without a vehicle. The morning gaps remained approximately **3.1** and **3.7 percentage points**, respectively, when matching within routes and school dates. The afternoon results did not show the same pattern.

The evidence supports a morning service disparity in this sample. It does not establish that individual students in these tracts ride transit more often or that unreliable buses caused school lateness. Stop-event totals describe scheduled service, and one trip can contribute multiple observations.

For interpretation, the estimates describe a specific period and a set of stop events. The analysis does not estimate statistical significance, and Census sampling error is not propagated into the comparisons.

## 13. Route and stop tables for the Explore Routes & Stops page
### Box 22 · Figures for every route and every route-stop pair
The website's second page lists each bus route and the Baltimore City timepoint stops it serves, for the morning, the afternoon, or both. It uses the same city stop events (`city`) and the same `metrics()` function as the map, so the totals agree with the map.

For each route, and each stop on each route, the box counts:
* **Classified stop events** and the share **canceled or more than 7 minutes late** (the same outcome as the claim).
* **Scheduled trips** and **whole trips canceled**. A stop event is one trip at one stop, so one canceled trip adds a canceled event at every stop it would have served. A trip counts as a whole canceled trip only when every one of its city timepoint stops was canceled.
* **Monitoring coverage**: the share of scheduled stop events the tracker was watching.
* **Missing vehicle records**: trips that never appeared in the real-time feed. A missed trip and a bus with broken tracking look the same, so these stay out of the rate and are shown separately.
* **Possible drive-pasts** from the `-stops` files: the bus averaged at least 15 mph between the two position reports around the stop and the feed never showed it stopping. This is inferred, the tracker cannot see riders, and these files start on November 2.
* **Mostly disrupted school dates**: dates on which half or more of the stop's classified events were canceled or more than 7 minutes late, so a high rate from a handful of events can be told apart from a pattern that repeats day after day. Fewer than 20 classified events is flagged as a small sample (the same cutoff as the map).


In [ ]:
# Route and stop tables for the "Explore Routes & Stops" page. They use the same Baltimore
# City stop events (`city`) and the same metrics() as the map, so every total agrees.
SMALL_SAMPLE = 20      # fewer classified events than this = "small sample" (same rule as the map)

city_rs = city.assign(route_id=city.route_id.astype(str))

# Possible drive-pasts come from the separate "-stops" files: for every bus that passed a stop,
# the tracker checks the 30-second position reports and records whether it stopped, drove past
# without stopping, or could not tell ("unclear"). These files start on November 2 (afternoon).
stop_files = sorted((repo / "data/school-commute-raw").glob("*-stops.csv.gz"))
behavior = pd.concat((pd.read_csv(f, dtype={"stop_id": str, "route_id": str}) for f in stop_files),
                     ignore_index=True)
behavior = behavior.loc[behavior.stop_id.isin(stop_tract)]       # city stops only, like `city`

def r1(value):
    return None if value is None else round(float(value), 1)

def trip_all(rows, outcome):
    """Trips on which every scheduled stop event in `rows` had this outcome."""
    return rows.outcome.eq(outcome).groupby([rows.service_date, rows.trip_id]).all().sum()

def table_row(rows, visits):
    """One route or route-stop row for one commute window."""
    m = metrics(rows)
    known = rows.loc[rows.classified]
    by_date = known.groupby("service_date").disrupted.mean().ge(0.5)   # half or more of that day's events
    seen = int(visits.visits.sum())
    return {
        "scheduled": m["scheduled"], "monitored": m["monitored"], "coverage": r1(m["coverage"]),
        "n": m["n"], "failed": r1(m["failed"]), "on": r1(m["on"]),
        "cancel": m["cancel"], "late": m["late"], "early": m["early"],
        "missing": m["missing"], "skipped": m["skipped"], "unclassified": m["unclassified"],
        "trips": int(rows.groupby(["service_date", "trip_id"]).ngroups),   # one bus run on one date
        "trips_canceled": int(trip_all(rows, "canceled")),             # every stop on the trip canceled
        "trips_missing": int(trip_all(rows, "no_realtime_record")),     # no vehicle record for the whole trip
        "dates": int(len(by_date)), "dates_disrupted": int(by_date.sum()),
        "visits": seen, "drove_past": int(visits.drove_past.sum()), "unclear": int(visits.unclear.sum()),
        "drive_past_pct": r1(100 * visits.drove_past.sum() / seen) if seen else None,
    }

windows = {"all": (city_rs, behavior),
           "am": (city_rs[city_rs.window.eq("am")], behavior[behavior.window.eq("am")]),
           "pm": (city_rs[city_rs.window.eq("pm")], behavior[behavior.window.eq("pm")])}
route_names = city_rs.groupby("route_id").route_name.last().astype(str)

# Split the records once by route and by route + stop, then fill each table row.
groups = {key: (dict(tuple(ev.groupby("route_id"))), dict(tuple(bh.groupby("route_id"))),
                dict(tuple(ev.groupby(["route_id", "stop_id"]))), dict(tuple(bh.groupby(["route_id", "stop_id"]))))
          for key, (ev, bh) in windows.items()}
no_visits = behavior.iloc[:0]

routes = []
for route_id, name in route_names.items():
    entry = {"id": route_id, "name": name}
    for key, (by_route, visits_by_route, _, _) in groups.items():
        rows = by_route.get(route_id)
        entry[key] = None if rows is None else table_row(rows, visits_by_route.get(route_id, no_visits))
    routes.append(entry)

route_stops = []
for route_id, stop_id in sorted(groups["all"][2]):
    entry = {"route": route_id, "stop": stop_id}
    for key, (_, _, by_pair, visits_by_pair) in groups.items():
        rows = by_pair.get((route_id, stop_id))
        entry[key] = None if rows is None else table_row(rows, visits_by_pair.get((route_id, stop_id), no_visits))
    route_stops.append(entry)

routes_payload = {
    "dates": payload["dates"], "small_sample": SMALL_SAMPLE,
    "drive_past_dates": {"first": str(behavior.service_date.min()), "last": str(behavior.service_date.max())},
    "city": {key: table_row(ev, bh) for key, (ev, bh) in windows.items()},
    "routes": routes, "route_stops": route_stops,
    "stop_names": {s: stops[s]["name"] for s in city_rs.stop_id.unique()},
}
routes_json = json.dumps(routes_payload, allow_nan=False, separators=(",", ":"))
Path("Baltimore_City_MTA_Routes_Stops.json").write_text(routes_json)
print(f"Exported {len(routes)} routes and {len(route_stops)} route-stop pairs "
      f"({len(routes_json) / 1e3:,.0f} KB).")


### Box 23 · Check the route tables against the map, and show the worst morning routes
The route rows must add up to the citywide map figures in each commute window (the `assert` stops the notebook if they do not). The second table lists the ten morning routes with the highest share canceled or more than 7 minutes late, among routes with at least 20 classified morning events.


In [ ]:
# Check: the route rows must add up to the citywide map figures, window by window.
check_rows = []
for key, label in [("all", "Both commutes"), ("am", "Morning"), ("pm", "Afternoon")]:
    rows = [r[key] for r in routes if r[key]]
    check_rows.append({"Window": label,
                       "Map: classified events": payload[key]["city"]["n"],
                       "Routes: classified events": sum(r["n"] for r in rows),
                       "Map: canceled or >7 min late (%)": round(payload[key]["city"]["failed"], 1),
                       "Routes: canceled or >7 min late (%)": round(100 * sum(r["cancel"] + r["late"] for r in rows)
                                                                    / sum(r["n"] for r in rows), 1)})
check_routes = pd.DataFrame(check_rows).set_index("Window")
assert (check_routes["Map: classified events"] == check_routes["Routes: classified events"]).all()
display(check_routes)

# The ten morning routes with the highest share canceled or more than 7 minutes late
# (routes with at least SMALL_SAMPLE classified morning events).
morning = pd.DataFrame([{"Route": r["name"], "Classified events": r["am"]["n"],
                         "Canceled or >7 min late (%)": r["am"]["failed"],
                         "Whole trips canceled": r["am"]["trips_canceled"], "Scheduled trips": r["am"]["trips"],
                         "Missing vehicle records": r["am"]["missing"], "Monitored (%)": r["am"]["coverage"],
                         "Possible drive-pasts (%)": r["am"]["drive_past_pct"]}
                        for r in routes if r["am"] and r["am"]["n"] >= SMALL_SAMPLE])
display(morning.sort_values("Canceled or >7 min late (%)", ascending=False).head(10).set_index("Route"))


## 14. Follow-up question: where do these conditions overlap?
**Question.** Do neighborhoods with less dependable bus service also have higher high school withdrawal rates?

This is a separate, area-level question. The claim above compares bus stop events by census tract. This section compares **areas**: Baltimore's 55 Community Statistical Areas (CSAs), the clusters of neighborhoods that the Baltimore Neighborhood Indicators Alliance (BNIA-JFI) uses for its Vital Signs indicators. A bus stop's location does not tell us where its riders live or go to school, and the stop data cannot tell which riders are students. So the boxes below can show *where conditions overlap*, not that late buses caused anyone to withdraw.

The four map measures:
* **High school withdrawal rate** (BNIA "High School Dropout/Withdrawal Rate"), school year 2022–23.
* **Households without a vehicle** (BNIA, from the American Community Survey 2019–2023).
* **Family households below the poverty line** (BNIA, ACS 2019–2023).
* **Bus disruption rate**: our own stop events, grouped by the CSA each stop sits in.

High school completion (2022–23) is shown as a separate figure, not as a map layer.

### Box 24 · Load BNIA's official CSA layers
BNIA publishes each Vital Signs indicator as a layer on its ArcGIS Online site (the data behind https://bniajfi.org/vital_signs/data_downloads/). On November 23, 2026 the tracker downloaded an exact copy of the layers it uses into the repository (`data/reference/bnia/raw/`, written by `scripts/fetch_bnia_csa.py`). This box reads that dated copy, so the education figures stay fixed while the bus data keeps updating. Box 25 then checks the copy against BNIA's live layers.

In [ ]:
# The dated copy of BNIA's layers (commit that saved them on Nov 23, 2026).
BNIA_COMMIT = "1bd6b1c3192954c48215e69eecbb782f662f5cb6"
BNIA_LAYERS = {   # file in the copy: what it holds
    "Community_Statistical_Areas_2020_0.geojson": "CSA 2020 boundaries",
    "Drop_0.geojson": "High School Dropout/Withdrawal Rate, by CSA",
    "Compl_0.geojson": "High School Completion Rate, by CSA",
    "Hhpov_0.geojson": "Family households below the poverty line, by CSA",
    "Novhcl_0.geojson": "Households with no vehicle available, by CSA",
    "Drop_1.geojson": "High School Dropout/Withdrawal Rate, citywide",
    "Compl_1.geojson": "High School Completion Rate, citywide",
    "Hhpov_1.geojson": "Family households below the poverty line, citywide",
    "Novhcl_1.geojson": "Households with no vehicle available, citywide",
    "manifest.json": "download date, layer addresses, field names and edit dates",
}
local = repo / "data/reference/bnia/raw"
bnia = {}
for name in BNIA_LAYERS:
    if (local / name).exists():                       # automatic website build: files are in the checkout
        text = (local / name).read_text()
    else:                                             # Colab: fetch the dated copy from GitHub
        url = f"https://raw.githubusercontent.com/TbilKitty/MTA_Accountability_Tracker/{BNIA_COMMIT}/data/reference/bnia/raw/{name}"
        target = root / name
        if not target.exists():
            download(url, target)
        text = target.read_text()
    bnia[name.rsplit(".", 1)[0]] = json.loads(text)

manifest = bnia.pop("manifest")
print(f"BNIA layers downloaded {manifest['downloaded_utc']} from {manifest['source']}")
for key, info in manifest["layers"].items():
    if key.split("_")[0] in ("Community", "Drop", "Compl", "Hhpov", "Novhcl"):
        edited = pd.to_datetime(info["data_last_edit"], unit="ms").date() if info["data_last_edit"] else "?"
        print(f"  {info['layer_name']:<85} {info['features']:>3} areas · data last edited {edited}")

### Box 25 · Check the definitions, years and identifiers
Each BNIA layer names its years in its field aliases (for example `drop23` = "2022-2023"). The box prints the definition BNIA attaches to each layer, confirms the latest year of each measure, and checks that every layer uses the same 55 CSA names (`CSA2020`), which is the identifier BNIA uses to join its tables. One CSA is spelled two ways ("Oliver/Johnson Square" in the indicator layers, "Oliver/Johnston Square" on the boundary layer); the box fixes that one spelling and confirms it is the same area through BNIA's 2010 name, Greenmount East. BNIA does not publish the student or household counts behind these percentages, and none of the latest values is blank. The check at the end re-reads BNIA's live layers (values only) and reports whether BNIA has changed them since the dated copy.

In [ ]:
import html as html_lib, re

# Measure key -> (BNIA layer, field for the latest year, label used on the page).
MEASURES = {
    "withdrawal": ("Drop_0", "drop23", "High school withdrawal rate"),
    "completion": ("Compl_0", "compl23", "High school completion rate"),
    "poverty":    ("Hhpov_0", "hhpov23", "Family households below the poverty line"),
    "no_vehicle": ("Novhcl_0", "novhcl23", "Households without a vehicle"),
}
period = {}
for key, (layer, field, label) in MEASURES.items():
    meta = manifest["layers"][layer]
    period[key] = meta["fields"][field]               # e.g. "2022-2023"
    later = [f for f in meta["fields"] if f.rstrip("0123456789") == field.rstrip("0123456789") and f > field]
    assert not later, f"{layer} has a newer year than {field}: {later}"
    print(f"{label}: latest = {field} ({period[key]})")

for layer in ("Drop_0", "Compl_0", "Hhpov_0", "Novhcl_0"):
    path = local / f"{layer}.meta.json"
    meta = json.loads(path.read_text()) if path.exists() else json.loads(urllib.request.urlopen(
        f"https://raw.githubusercontent.com/TbilKitty/MTA_Accountability_Tracker/{BNIA_COMMIT}/data/reference/bnia/raw/{layer}.meta.json").read())
    words = re.sub(r"\s+", " ", re.sub("<[^>]+>", " ", html_lib.unescape(meta.get("description") or ""))).strip()
    print(f"\n{meta['name']}\n  {words}")

# One table: 55 CSAs x the four BNIA measures, joined on the CSA name.
# BNIA's indicator layers spell one CSA "Oliver/Johnson Square"; its boundary layer says
# "Oliver/Johnston Square". Both are the CSA that BNIA's 2010 field calls "Greenmount East".
NAME_FIXES = {"Oliver/Johnson Square": "Oliver/Johnston Square"}
def layer_values(layer, field):
    return {NAME_FIXES.get(f["properties"]["CSA2020"], f["properties"]["CSA2020"]): f["properties"][field]
            for f in bnia[layer]["features"]}
assert {f["properties"]["CSA2010"] for f in bnia["Drop_0"]["features"]
        if f["properties"]["CSA2020"] in NAME_FIXES} == {"Greenmount East"}

csa_table = pd.DataFrame({key: layer_values(layer, field) for key, (layer, field, _) in MEASURES.items()})
names_by_layer = {layer: set(layer_values(layer, field)) for layer, field, _ in MEASURES.values()}
assert len({frozenset(s) for s in names_by_layer.values()}) == 1, "CSA names differ between layers"
boundaries = {f["properties"]["CSA2020"]: f for f in bnia["Community_Statistical_Areas_2020_0"]["features"]}
print(f"\n{len(csa_table)} CSAs with indicator values; boundary layer has {len(boundaries)} shapes")
print("Boundary shapes without indicators:", sorted(set(boundaries) - set(csa_table.index)))
assert set(csa_table.index) <= set(boundaries)
print("Blank values per measure:", csa_table.isna().sum().to_dict())

# Compare with BNIA's live layers (values only; skipped if the BNIA site cannot be reached).
live_url = "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/{}/FeatureServer/0/query?where=1%3D1&outFields=CSA2020,{}&returnGeometry=false&f=json"
try:
    changed = []
    for key, (layer, field, label) in MEASURES.items():
        rows = json.loads(urllib.request.urlopen(live_url.format(layer.split("_")[0], field), timeout=60).read())["features"]
        live = {r["attributes"]["CSA2020"]: r["attributes"][field] for r in rows}
        changed += [(label, c) for c, v in live.items() if v is None or abs(v - csa_table.at[c, key]) > 1e-9]
    print("BNIA live layers match the dated copy." if not changed else f"BNIA has changed {len(changed)} values since the copy: {changed[:5]}")
except Exception as error:
    print(f"Could not reach BNIA's live layers ({error}); using the dated copy.")

### Box 26 · Flag withdrawal rates that are too unstable to compare
Because BNIA does not publish how many students are behind each withdrawal rate, a rate for an area with few high school students can swing wildly from year to year. This box applies one written-down rule: a 2022–23 withdrawal rate is marked **Insufficient data** for the comparisons if it differs by more than 20 percentage points from the same area's average for the two prior school years. The value is still shown when you click the area, with that label. The citywide rate is printed for scale.

In [ ]:
UNSTABLE_POINTS = 20
prior = pd.DataFrame({y: layer_values("Drop_0", y) for y in ("drop21", "drop22")})
csa_table["withdrawal_prior_avg"] = prior.mean(axis=1)
csa_table["withdrawal_unstable"] = (csa_table.withdrawal - csa_table.withdrawal_prior_avg).abs() > UNSTABLE_POINTS
city_values = {key: bnia[layer.replace("_0", "_1")]["features"][0]["properties"][field]
               for key, (layer, field, _) in MEASURES.items()}
print("Citywide values:", {k: round(v, 1) for k, v in city_values.items()})
display(csa_table.loc[csa_table.withdrawal_unstable, ["withdrawal", "withdrawal_prior_avg"]].round(1))

### Box 27 · Place each Baltimore City bus stop in a CSA and add up its events
Each stop's published coordinate is matched to the CSA boundary it falls inside. A CSA's disruption rate is then calculated from **summed event counts**: all classified stop events at its stops, and of those, the ones canceled or more than 7 minutes late. (Averaging stop percentages would let a stop with 3 events count as much as a stop with 300.) A CSA with fewer than 20 classified events in a commute window is marked Insufficient data, the same rule as the rest of the site.

In [ ]:
csa_shapes = [(name, make_valid(shape(f["geometry"]))) for name, f in boundaries.items() if f["geometry"]]
csa_tree = STRtree([g for _, g in csa_shapes])
stop_csa = {}
for stop_id in city.stop_id.unique():
    hits = csa_tree.query(Point(stops[stop_id]["lon"], stops[stop_id]["lat"]), predicate="within")
    if len(hits) == 1:
        stop_csa[stop_id] = csa_shapes[hits[0]][0]
city["csa"] = city.stop_id.map(stop_csa)
print(f"{len(stop_csa):,} of {city.stop_id.nunique():,} city stops placed in a CSA "
      f"({city.csa.notna().mean():.1%} of city stop events)")

SMALL_SAMPLE = 20
csa_service = {}
for window in ("am", "pm"):
    rows = city.loc[city.window.eq(window) & city.csa.notna()]
    csa_service[window] = {name: metrics(group) for name, group in rows.groupby("csa")}
    rates = pd.Series({n: m["failed"] for n, m in csa_service[window].items() if m["n"] >= SMALL_SAMPLE})
    csa_table[f"disruption_{window}"] = rates
    print(f"{window.upper()}: {rates.size} CSAs with {SMALL_SAMPLE}+ classified events; "
          f"rates from {rates.min():.1f}% to {rates.max():.1f}%")

### Box 28 · Highest-quarter cutoffs and the four-way overlap
"Highest quarter" means among the highest 25% of Baltimore City CSAs **that have data for that measure** in the period shown: a CSA is in it when its value is at or above the 75th percentile of those CSAs. (For completion, where lower is worse, the lowest quarter is used: at or below the 25th percentile.) Areas marked Insufficient data are left out of both the cutoff and the overlap for that measure. The overlap is the list of CSAs in the highest quarter on all four measures; it is computed separately for the morning and afternoon bus layer.

In [ ]:
def cutoff(values, low=False):
    values = values.dropna()
    return float(values.quantile(0.25 if low else 0.75)), int(values.size)

comparable = csa_table.assign(withdrawal=csa_table.withdrawal.mask(csa_table.withdrawal_unstable))
cutoffs = {key: cutoff(comparable[key]) for key in ("withdrawal", "poverty", "no_vehicle", "disruption_am", "disruption_pm")}
cutoffs["completion"] = cutoff(comparable.completion, low=True)
for key, (value, n) in cutoffs.items():
    print(f"{key:<14} cutoff {value:6.1f}%   ({n} CSAs with data)")

overlap = {}
for window in ("am", "pm"):
    keys = ["withdrawal", "no_vehicle", "poverty", f"disruption_{window}"]
    meets = pd.DataFrame({k: comparable[k].ge(cutoffs[k][0]) for k in keys})
    overlap[window] = sorted(meets.index[meets.all(axis=1)])
    print(f"\n{window.upper()}: {len(overlap[window])} CSAs in the highest quarter on all four:", overlap[window] or "none")
    print("  in the highest quarter on 3 of 4:", sorted(meets.index[meets.sum(axis=1).eq(3)]))

### Box 29 · An area-level association, stated carefully
Across the CSAs that have both a usable withdrawal rate and a bus disruption rate, do the two rise together? The box reports a Spearman rank correlation (from −1 to +1; 0 means no consistent ordering), the number of CSAs it uses, and a permutation check: it shuffles the disruption rates among the CSAs 10,000 times and counts how often a shuffle gives a correlation at least as far from zero. A small share (say under 5%) means the ordering is unlikely to be a coincidence of which areas happen to sit together; it still says nothing about cause. This is an **area-level association across different periods** (withdrawal 2022–23, buses autumn 2026): it describes places, not students, and cannot show that bus service affects withdrawal.

In [ ]:
import numpy as np
rng = np.random.default_rng(2026)                     # fixed seed: the same result every run
association = {}
for window in ("am", "pm"):
    both = comparable[["withdrawal", f"disruption_{window}"]].dropna()
    ranks = both.rank().to_numpy()
    rho = np.corrcoef(ranks[:, 0], ranks[:, 1])[0, 1]  # Spearman = correlation of the ranks
    shuffled = np.array([np.corrcoef(ranks[:, 0], rng.permutation(ranks[:, 1]))[0, 1] for _ in range(10_000)])
    share = float((np.abs(shuffled) >= abs(rho) - 1e-12).mean())
    association[window] = {"rho": round(float(rho), 2), "n": int(len(both)), "permutation_share": round(share, 4)}
    print(f"{window.upper()}: Spearman rank correlation {rho:+.2f} across {len(both)} CSAs; "
          f"{share:.1%} of 10,000 shuffles were as far from zero")

### Box 30 · Spot-check a few areas against BNIA's own numbers
Three CSAs, with the values exactly as BNIA's layers store them next to the values this notebook will publish. They must match to the decimal.

In [ ]:
spot = ["Sandtown-Winchester/Harlem Park", "Canton", "Cherry Hill"]
check = pd.DataFrame({key: [layer_values(layer, field)[c] for c in spot] for key, (layer, field, _) in MEASURES.items()}, index=spot)
assert (check.round(6) == csa_table.loc[spot, list(MEASURES)].round(6)).all().all()
display(check.round(2))

### Box 31 · Export the map files
Writes three files for the website:
* `bnia_csa_indicators_2022-23.json`: the **versioned education and Census dataset** (BNIA values by CSA, with source links, the BNIA data date, each indicator's year and the geography). It does not change when the bus data refreshes.
* `csa_overlap.json`: the bus disruption counts by CSA and window, the cutoffs, the overlap lists and the association. This one refreshes with the bus data.
* `csa_map.svg`: the CSA shapes, drawn the same way as Box 13.

In [ ]:
def r(value, digits=2):
    return None if value is None or pd.isna(value) else round(float(value), digits)

indicator_source = {
    "withdrawal": "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/Drop/FeatureServer/0",
    "completion": "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/Compl/FeatureServer/0",
    "poverty": "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/Hhpov/FeatureServer/0",
    "no_vehicle": "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/Novhcl/FeatureServer/0",
}
education = {
    "version": "bnia-csa-2022-23-v1",
    "downloaded": manifest["downloaded_utc"][:10],
    "geography": "Community Statistical Areas, 2020 boundaries (55 areas with indicator values); joined on the CSA2020 name",
    "boundaries": "https://services1.arcgis.com/mVFRs7NF4iFitgbY/ArcGIS/rest/services/Community_Statistical_Areas_2020/FeatureServer/0",
    "publisher": "Baltimore Neighborhood Indicators Alliance–Jacob France Institute (BNIA-JFI), Vital Signs; https://bniajfi.org/vital_signs/data_downloads/",
    "unstable_rule": f"Withdrawal rate marked Insufficient data if it differs by more than {UNSTABLE_POINTS} points from the area's 2020-21/2021-22 average",
    "measures": {key: {
        "label": label, "bnia_field": field, "period": period[key],
        "period_type": "school year" if key in ("withdrawal", "completion") else "ACS 5-year estimate",
        "source": indicator_source[key], "layer_name": manifest["layers"][layer]["layer_name"],
        "data_last_edited": str(pd.to_datetime(manifest["layers"][layer]["data_last_edit"], unit="ms").date()),
        "citywide": r(city_values[key]),
    } for key, (layer, field, label) in MEASURES.items()},
    "areas": {name: {"withdrawal": r(row.withdrawal), "withdrawal_unstable": bool(row.withdrawal_unstable),
                     "withdrawal_prior_avg": r(row.withdrawal_prior_avg), "completion": r(row.completion),
                     "poverty": r(row.poverty), "no_vehicle": r(row.no_vehicle)}
              for name, row in csa_table.iterrows()},
}

def compact(m):
    keep = ("n", "failed", "cancel", "late", "early", "on", "avg", "monitored", "scheduled")
    return {k: (r(m[k]) if isinstance(m[k], float) else m[k]) for k in keep}

csa_payload = {
    "dates": payload["dates"], "small_sample": SMALL_SAMPLE,
    "school_dates": {w: int(city.loc[city.window.eq(w)].service_date.nunique()) for w in ("am", "pm")}, "education_version": education["version"],
    "service": {w: {name: compact(m) for name, m in csa_service[w].items()} for w in ("am", "pm")},
    "cutoffs": {k: {"value": r(v), "areas": n, "direction": "low" if k == "completion" else "high"} for k, (v, n) in cutoffs.items()},
    "overlap": overlap, "association": association,
    "stops_placed": len(stop_csa), "city_stops": int(city.stop_id.nunique()),
}

# CSA shapes on the same projection idea as Box 13, fitted to the CSA outlines.
cxmin, cymin, cxmax, cymax = unary_union([g for _, g in csa_shapes]).bounds
cscale = min(560 / ((cxmax - cxmin) * X_STRETCH), 535 / (cymax - cymin))
CSA_W, CSA_H = round((cxmax - cxmin) * cscale * X_STRETCH + 40), round((cymax - cymin) * cscale + 40)

def csa_ring(ring):
    pts = ((20 + (x - cxmin) * cscale * X_STRETCH, 20 + (cymax - y) * cscale) for x, y in ring.coords)
    return "M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in pts) + "Z"

csa_svg = (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {CSA_W} {CSA_H}" role="group">' + "".join(
    f'<path d="{"".join(csa_ring(rg) for rg in outline_rings(g.simplify(0.00005)))}" data-csa="{escape(name, quote=True)}" '
    f'tabindex="0" role="button" aria-label="{escape(name, quote=True)}"/>'
    for name, g in csa_shapes) + "</svg>")

Path("bnia_csa_indicators_2022-23.json").write_text(json.dumps(education, indent=1, allow_nan=False))
Path("csa_overlap.json").write_text(json.dumps(csa_payload, separators=(",", ":"), allow_nan=False))
Path("csa_map.svg").write_text(csa_svg)
print(f"Exported {len(education['areas'])} CSAs; map {CSA_W} x {CSA_H}, {len(csa_svg) / 1e3:,.0f} KB")